In [1]:
import oracledb
import sys

from sqlalchemy.dialects.oracle.base import OracleDialect
from sqlalchemy import create_engine, text, MetaData, Table, func

from geoalchemy2.types import Geometry

In [2]:
sys.modules["cx_Oracle"] = oracledb
oracledb.version = "8.3.0"

oracledb.init_oracle_client()

In [3]:
with open("oracle-dsn.txt") as f:
    src_engine = create_engine(f.read())

In [4]:
with src_engine.connect() as conn:
    _table = 'sde.sde_version'

    if conn.dialect.name == 'oracle':
        _table = 'sde.version'        

    major, minor, bugfix = conn.execute(
        text(f"SELECT major, minor, bugfix FROM {_table}"),
        
    ).fetchone()

print(f"ArcSDE: {major}.{minor}.{bugfix}")

ArcSDE: 10.1.0


In [5]:
class STGeometry(Geometry):
    name = "st_geometry"

    from_text = "sde.st_geomfromtext"

    as_binary = "sde.st_asbinary"

    def __init__(
            self, 
            geometry_type = "GEOMETRY", 
            srid = -1, 
            dimension = None, 
            spatial_index = True, 
            nullable = True, 
            _spatial_index_reflected=None
        ):
        super().__init__(
            geometry_type=geometry_type, 
            srid=srid, 
            dimension=dimension, 
            spatial_index=spatial_index, 
            use_N_D_index=False,
            use_typmod=False,
            nullable=nullable, 
            _spatial_index_reflected=_spatial_index_reflected
        )

    def get_col_spec(self, **kw):
        return "ST_GEOMETRY"

    def column_expression(self, col):
        """
        No SELECT, converte o objeto ESRI para WKB usando a função do SDE.
        """
        return func.sde.st_asbinary(col)

    def bind_expression(self, bindvalue):
        return func.sde.st_geomfromtext(bindvalue)

##########

sde = STGeometry()
OracleDialect.ischema_names[sde.get_col_spec()] = sde.__class__

metadata = MetaData(schema='recmin')
tabela = Table('rm_layer', metadata, autoload_with=src_engine)

In [9]:
geometry = [col for col in tabela.columns][27]
geometry.type

STGeometry(dimension=2)

In [10]:
dir(geometry.type)

['Comparator',
 'ElementType',
 '__annotations__',
 '__class__',
 '__class_getitem__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__orig_bases__',
 '__parameters__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__slots__',
 '__str__',
 '__subclasshook__',
 '__visit_name__',
 '__weakref__',
 '_cached_bind_processor',
 '_cached_custom_processor',
 '_cached_literal_processor',
 '_cached_result_processor',
 '_compare_type_affinity',
 '_compiler_dispatch',
 '_compiler_dispatcher',
 '_default_dialect',
 '_dialect_info',
 '_gen_dialect_impl',
 '_generate_compiler_dispatch',
 '_generic_type_affinity',
 '_has_bind_expression',
 '_has_column_expression',
 '_is_array',
 '_is_table_value',
 '_is_tuple_type',
 '_is_type_decorator',
 '_isnull',
 '_original_compile

In [8]:
with src_engine.connect() as conn:
    data = conn.execute(tabela.select().limit(10)).fetchall()

data

[(93362, 181682, 'Oracle', 'Levantamento em carta 1:250.000', 'Fazenda Pedra', 'Itiúba', 'BA', 'A Definir', datetime.datetime(2006, 11, 24, 0, 0), None, None, None, None, 'OUTROS', None, None, None, 19533, 'São Francisco', 'Indeterminado', 'Indeterminado', None, None, None, None, None, 'Mármore', b'\x01\x01\x00\x00\x00\xd0\xfe\xdc\xd7N\xe4C\xc0`(5\xe1\x82\xdf$\xc0', None, None, 'Material de uso na construção civil', None, None, None, Decimal('-39.7836561'), Decimal('-10.43654541'), 'WGS84'),
 (93363, 181683, 'Oracle', 'Levantamento em carta 1:250.000', 'Fazenda das Flores', 'Monte Santo', 'BA', 'A Definir', datetime.datetime(2006, 11, 24, 0, 0), None, None, None, None, 'OUTROS', None, None, None, 19534, 'São Francisco', 'Indeterminado', 'Indeterminado', 'Mina subterrânea (GPS sem sinal)', None, None, None, None, 'Mármore', b'\x01\x01\x00\x00\x00\x10\xc6\xc4\xf0\x15\xceC\xc0`E\xec\x97xH$\xc0', None, None, 'Material de uso na construção civil', None, None, None, Decimal('-39.61004457'), 